# Packages import

In [12]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Ceneo scraper

1. Provide URL address of products opinions webpage

In [13]:

product_code = '100714868'
page = 1
url = f'https://www.ceneo.pl/{product_code}/opinie-{page}'

2. Send the request to provide url address

In [14]:
response = requests.get(url)
print(response.status_code)

200


3. If status code is OK, fetch product name

In [15]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [16]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text()
print(product_name)

Psi Patrol Kosmopieski Ratują Zatokę Przygód (Digital) - Klucz aktywacyjny


4. If status code is OK, fetch all opinions from requested webpage

In [17]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [18]:
all_opinions = page_dom.find_all('div', class_='js_product-review')
opinions = [r for r in all_opinions if 'user-post--highlight' not in r.get('class', [])]
print(type(opinions))
print(len(opinions))

<class 'list'>
10


5. For all fetched opinions, parse them to extract relevant data

In [19]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion['data-entry-id'],
        'author': opinion.select_one('span.user-post__author-name').get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recomendation > em').get_text().strip(),
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'like': opinion.select_one('span[id^="votes-yes"]').get_text().strip(),
        'dislike': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publishing_date': opinion.select_one('span.user-post__published > time:nth-child(1)')['datetime'],
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)')['datetime'].strip() if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None
    }
    all_opinions.append(single_opinion)


6. Check if there is next page with opinions 

In [20]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page +=1


8. Save obtained opinions

In [21]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [22]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4,ensure_ascii=False)
